# Phase 4: LLM evaluation with Ollama (gold vs retrieved contexts, LLM-as-judge)

In [ ]:
# Setup: clone the repo and copy the datasets (same layout as retrieval-benchmark-full).
# The Hugging Face token comes from Kaggle Secrets (Add-ons > Secrets > HF_TOKEN); never paste it here.
!pip install -q "datasets<4.0.0" qdrant_client
!git clone https://github.com/Scrowbin/science-document-viewer
!cd science-document-viewer && git checkout feat/app-improvements && git log --oneline -1
!pip install -q -r science-document-viewer/backend/requirements.txt
import os
from kaggle_secrets import UserSecretsClient
os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
os.environ["DEBUG"] = "True"  # backend settings need DEBUG or a SECRET_KEY
!cp -r /kaggle/input/datasets/kristiko/qasper-official /kaggle/working/science-document-viewer/eval/data/
!mv /kaggle/working/science-document-viewer/eval/data/qasper-official /kaggle/working/science-document-viewer/eval/data/qasper_official
!mkdir -p /kaggle/working/science-document-viewer/eval/pdfs
!cp -r /kaggle/input/datasets/kristiko/peerqa /kaggle/working/science-document-viewer/eval/pdfs/peerqa
!find /kaggle/working/science-document-viewer/eval/pdfs/peerqa -name "*.pdf" | wc -l   # expect about 107

In [ ]:
# Install and start Ollama on the GPU, then pull the models (about 15 GB in total)
!curl -fsSL https://ollama.com/install.sh | sh
import subprocess, time
subprocess.Popen(["ollama", "serve"], stdout=open("/kaggle/working/ollama.log", "w"), stderr=subprocess.STDOUT)
time.sleep(5)
for m in ["llama3.2:3b", "qwen2.5:1.5b", "phi4-mini", "qwen3:4b", "gemma3:4b", "qwen2.5:14b"]:
    !ollama pull {m}

In [ ]:
S = 'science-document-viewer/eval/phases/phase_4_generation/scripts/evaluate_llm_generation.py'
# smoke test
!python {S} --cohort C-RET --condition gold --limit 2 --models llama3.2:3b

1) Validate the judge on the 100 human-labelled answers before trusting it.

In [ ]:
!python {S} --judge-validate

2) Gold-evidence contexts (all 5 models, both cohorts). Resumable.

In [ ]:
!python {S} --condition gold

3) Retrieved contexts. Set PIPELINE to the arm chosen from Phase 3 (format `embedder:arm`, e.g. `qwen:convex+bgem3`, `e5:dense`).

In [ ]:
PIPELINE = 'qwen:dense'  # <- replace with the Phase 3 choice
!python {S} --condition retrieved --pipeline {PIPELINE}

4) Judge every answer (qwen2.5:14b).

In [ ]:
!python {S} --judge

In [ ]:
# Zip the results and download them from the Output tab, then unzip into the repo at the same path.
!cd science-document-viewer && zip -qr /kaggle/working/phase4_artifacts.zip eval/phases/phase_4_generation/artifacts